# Caformer Chat — Replica & Pipeline Walkthrough

Reproduces every step of the `/caformer/chat/` reply pipeline in a notebook so you can
introspect and modify each stage without the web view:

1. Receive a prompt; encode as bytes (vocab = 256)
2. **Dispatcher** — route to the right rules:
   - Action triggers (substring-driven heavier experts)
   - Exact prompt match against trained QRPairs
   - Fuzzy prenormalizer (Levenshtein on trained prompts)
   - Per-word composition
   - Fallback to random rules
3. Load the trained genome + positional output rules
4. Forward pass — `ca_forward_qkv` per token
5. Sample next token (temperature-0 = argmax)
6. Decode bytes back to text


In [ ]:
import os, sys, django
REPO = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, REPO)
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'velour.settings')
django.setup()
import numpy as np
from caformer.models import QRPair, TrainedModel
from caformer.transformer import ca_forward_qkv
from caformer.primitives import ca_softmax_sample
from caformer.qr_trainer import sample_positional, blob_to_genome
from caformer import views as cf_views

## 1. Pick a prompt and tokenize
Byte-level tokenizer; vocab = 256. Every UTF-8 byte → one token id.

In [ ]:
prompt = 'hi'
prompt_ids = list(prompt.encode('utf-8'))
print(f'prompt: {prompt!r}')
print(f'token ids: {prompt_ids}')

## 2. Dispatcher — find the right model for this prompt
Mirrors the priority order in `chat_reply` / `chat_reply_stream`.

In [ ]:
def dispatch(q):
    trig = cf_views._resolve_action_trigger(q)
    if trig:
        cfg, payload = trig
        if cfg.get('expert_slug') and TrainedModel.objects.filter(slug=cfg['expert_slug']).exists():
            return ('action', cfg['expert_slug'], payload or q, int(cfg.get('n_blocks', 1)))
    slug, n_blocks = cf_views._dispatch_qrpair_for_prompt(q)
    if slug:
        return ('exact', slug, q, n_blocks)
    norm = cf_views._normalize_prompt_to_trained(q)
    if norm:
        canonical, slug, n_b, dist = norm
        return (f'normalized(d={dist})', slug, canonical, n_b)
    return ('fallback', '', q, 2)

mode, slug, q_eff, n_blocks = dispatch(prompt)
print(f'dispatch: mode={mode}  slug={slug}  q_eff={q_eff!r}  n_blocks={n_blocks}')
if mode == 'exact':
    print(f'\\n→ Will run the trained {slug} positional pipeline.')

## 3. Load the trained genome
When dispatch returns a positional QRPair, we load **base rules from the QRPair** (not the TrainedModel snapshot — they drift on retrain) plus the **per-position output rules**.

In [ ]:
if slug:
    pair = QRPair.objects.get(deployed_slug=slug)
    genome = blob_to_genome(pair.best_genome_blob)
    pos_rules = pair.positional_output_rules()
    print(f'pair #{pair.pk}: {pair.prompt!r} → {pair.expected!r}')
    print(f'best_exact: {pair.best_exact}')
    print(f'genome rules: {list(genome.keys())}')
    print(f'positional output rules: {len(pos_rules)} (one per target byte)')
else:
    print('no trained slug; falling back to random rules')

## 4-5. Forward pass + sample, per token
This is the inner loop. At each tick, swap in the trained positional output rule for the current position.

In [ ]:
n_tokens = len(pos_rules) if slug else 8
temperature = 0.0
seed = 0xCAFE

block_template = [{k: genome[k] for k in ('q','k','v','score','mix','merge','mlp')}] * n_blocks

ctx = list(q_eff.encode('utf-8'))
out_ids = []
for tick in range(n_tokens):
    output_rule = pos_rules[tick] if tick < len(pos_rules) else genome['output']
    logits = ca_forward_qkv(
        ctx, n_blocks=n_blocks,
        embed_rule=genome['embed'],
        block_rules=block_template,
        norm_rule=genome['norm'],
        output_rule=output_rule,
        vocab_size=256)
    nxt, _ = ca_softmax_sample(logits, temperature=temperature, ca_seed=seed ^ (tick + 1))
    out_ids.append(int(nxt))
    ctx.append(int(nxt))

reply = bytes(out_ids).decode('latin-1', errors='replace')
print(f'output token ids: {out_ids}')
print(f'decoded reply:    {reply!r}')

## 6. Compare with the chat endpoint
Running the same prompt through the actual Django view should produce byte-identical output.

In [ ]:
import asyncio, json
from django.test import AsyncClient
from django.contrib.auth import get_user_model
async def hit_chat(p):
    c = AsyncClient()
    u = await get_user_model().objects.afirst()
    await c.aforce_login(u)
    r = await c.get(f'/caformer/chat/reply/?q={p}&temperature=0&n=8')
    return json.loads(r.content)
result = asyncio.run(hit_chat(prompt))
print(json.dumps(result, indent=2)[:400])

## 7. Per-word composition
When the whole prompt has no match but contains trained words, the chat composes a reply per-word and concatenates with spaces. Untrained words pass through verbatim.

In [ ]:
for p in ['hi', 'hi hey', 'hi friend hey', 'hellloo']:
    r = asyncio.run(hit_chat(p))
    print(f'{p!r:>22s}  dispatched={r["dispatched"]!r:>50s}  reply={r["reply"]!r}')

## 8. Train a new pair from this notebook
Same trainer the `✏ fix this` UI uses; auto-deploys on `best_exact`. Drop the cell below into the chat and the dispatcher routes to it automatically.

In [ ]:
from caformer.qr_trainer import train_pair_positional, PositionalTrainConfig
p = QRPair.objects.get_or_create(prompt='yo', expected='sup',
                                 defaults={'n_blocks': 1, 'label': 'notebook'})[0]
if not p.best_exact:
    cfg = PositionalTrainConfig(pop_size=24, gens_per_phase=24, polish_trials=120, max_seconds=120)
    train_pair_positional(p.pk, cfg=cfg)
    p.refresh_from_db()
print(f'yo → {p.best_output!r}  exact={p.best_exact}  deployed={p.deployed_slug!r}')